# Lecția 10 - Agenți AI în producție

În această lecție veți învăța **modele de producție** pentru agenții AI folosind Microsoft Agent Framework (Python). Acoperim:

- **Observabilitate** — adăugarea măsurării timpului și jurnalizării interacțiunilor agentului
- **Evaluare** — utilizarea unui agent evaluator pentru a puncta calitatea răspunsurilor
- **Gestionarea costurilor** — strategii pentru optimizarea tokenilor și selecția modelului

Scenariul este un **agent de turism** care ajută utilizatorii să planifice călătorii, cu monitorizare și evaluare integrate.


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Considerații pentru Producție

Mutarea agenților AI de la prototipuri la producție necesită o atenție atentă la trei piloni:

1. **Observabilitate** — Aveți nevoie de vizibilitate asupra a ceea ce face agentul, cât timp durează și ce instrumente apelează. Fără trasare și jurnalizare, depanarea problemelor din producție este aproape imposibilă.

2. **Evaluare** — Verificările automate de calitate asigură că răspunsurile agentului rămân precise, complete și utile pe termen lung. Un agent evaluator poate puncta răspunsurile în funcție de criterii definite.

3. **Gestionarea Costurilor** — Utilizarea tokenilor influențează direct costul. Strategii precum optimizarea promptului, selecția modelului și caching-ul ajută la menținerea cheltuielilor sub control fără a sacrifica calitatea.


## Construirea unui Agent Observabil

Definim unelte de călătorie și înfășurăm apelul agentului cu cronometrul astfel încât să putem monitoriza latența. În producție, ai integra cu OpenTelemetry sau un backend similar de urmărire.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Modele de evaluare

Un model comun în producție este să se utilizeze un al doilea agent ca **evaluator**. Evaluatorul evaluează răspunsul agentului primar în funcție de criterii prestabilite precum completitudinea, acuratețea și utilitatea.

Aceasta permite:
- Porți automate de calitate înainte ca răspunsurile să ajungă la utilizatori
- Detectarea regresiilor când prompturile sau modelele se modifică
- Monitorizarea continuă a performanței agentului în timp


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategii de management al costurilor

Controlul costurilor este esențial pentru agenții AI de producție. Iată strategii cheie:

| Strategie | Descriere |
|---|---|
| **Optimizarea promptului** | Păstrați instrucțiunile de sistem concise. Eliminați contextul redundant pentru a reduce tokenii de intrare. |
    "| **Selecția modelului** | Utilizați modele mai mici și mai ieftine (de exemplu, GPT-5-mini) pentru sarcini simple, cum ar fi clasificarea sau extragerea, și rezervați modelele mai mari pentru raționamente complexe. |\n",
| **Caching** | Stocați în cache rezultatele instrumentelor și interogările frecvente pentru a evita apelurile redundante la API. |
| **Bugete de tokeni** | Stabiliți limite pentru `max_tokens` pentru a preveni răspunsurile neașteptat de lungi. |
| **Batching** | Grupați mai multe interogări ale utilizatorilor într-un singur apel API, acolo unde este posibil. |

În practică, o abordare pe niveluri funcționează bine: direcționați cererile simple către un model rapid și ieftin și escaladați doar interogările complexe către unul mai capabil.


## Rezumat

În această lecție ai învățat cum să:

1. **Adaugi observabilitate** interacțiunilor agentului cu cronometrare și înregistrare, punând bazele pentru trasare și monitorizare.
2. **Evaluezi răspunsurile agentului** automat folosind un agent evaluator care notează completitudinea, acuratețea și utilitatea.
3. **Gestionezi costurile** prin optimizarea prompturilor, selecția modelului, cache și bugete de tokeni.

Aceste tipare de producție ajută la asigurarea faptului că agenții tăi AI sunt fiabili, măsurabili și rentabili la scară.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
